In [102]:
#README


In [103]:
#IMPORTS

import yfinance as yf
import pandas as pd
import numpy as np



In [104]:
#INPUTS

target = yf.Ticker("MSFT")


benchmark = "URTH"
start_date = "2020-12-31"
end_date = "2025-12-31"
period = "5y"
interval = "1wk"
return_calc = "linear" #linear / log

peer_group = ["ORCL", "PLTR", "PANW", "CRWD", "FTNT"]


In [105]:
#FUNCTIONS

def download_data(tickers: list, start_date: str, end_date:str, interval: str):
    """Download data for the given tickers from Yahoo Finance"""
    data = yf.download(tickers = tickers, start = start_date, end = end_date, interval = interval)
    return data

def save_data(data: pd.DataFrame):
    """Save data to csv file"""
    path = f"./data/{benchmark} + {peer_group}_{start_date}_-_{end_date}.csv"
    data.to_csv(path)

def extract_col(data: pd.DataFrame, field: str):
    """Extract columns from the downloaded dataframe"""
    column_data= data[field]
    return column_data

# def quick_beta(peer, benchmark):
#     cov = np.cov(peer, benchmark)[0, 1]
#     var = np.var(benchmark)
#     return cov / var


In [106]:
#CLOSE_DATA_COLLECTION

ticker_package = peer_group + [benchmark]
data_package = download_data(tickers= ticker_package, start_date= start_date, end_date= end_date, interval= interval)
save_data(data_package)
close_data = extract_col(data_package, field= "Close")
close_data.head()


[*********************100%***********************]  6 of 6 completed


Ticker,CRWD,FTNT,ORCL,PANW,PLTR,URTH
Date,,,,,,
2020-12-28,52.955002,29.705999,59.762871,59.231667,23.549999,103.073921
2021-01-04,55.932499,29.628000,58.776691,61.091667,25.200001,105.733055
2021-01-11,54.877499,29.306000,57.292908,60.811668,25.639999,104.119225
2021-01-18,55.880001,30.246000,55.976048,60.770000,32.580002,105.769722
2021-01-25,53.950001,28.950001,56.040966,58.458332,35.180000,102.230309


In [107]:
#BASIC DATA CLEARING

close_data.columns = close_data.columns.get_level_values(0)
close_data = close_data.dropna()
close_data.head()

Ticker,CRWD,FTNT,ORCL,PANW,PLTR,URTH
Date,,,,,,
2020-12-28,52.955002,29.705999,59.762871,59.231667,23.549999,103.073921
2021-01-04,55.932499,29.628000,58.776691,61.091667,25.200001,105.733055
2021-01-11,54.877499,29.306000,57.292908,60.811668,25.639999,104.119225
2021-01-18,55.880001,30.246000,55.976048,60.770000,32.580002,105.769722
2021-01-25,53.950001,28.950001,56.040966,58.458332,35.180000,102.230309


In [108]:
#LOG_RETURN_CALCULATION
if return_calc == "log":
    return_data = np.log(close_data / close_data.shift(1))
elif return_calc == "linear":
    return_data = (close_data / close_data.shift(1)) -1
else:
    raise ValueError("Return calculation must be either 'log' or 'linear'")

return_data = return_data.dropna()
return_data.head()

Ticker,CRWD,FTNT,ORCL,PANW,PLTR,URTH
Date,,,,,,
2021-01-04,0.056227,-0.002626,-0.016502,0.031402,0.070064,0.025798
2021-01-11,-0.018862,-0.010868,-0.025244,-0.004583,0.017460,-0.015263
2021-01-18,0.018268,0.032075,-0.022985,-0.000685,0.270671,0.015852
2021-01-25,-0.034538,-0.042849,0.001160,-0.038040,0.079804,-0.033463
2021-02-01,0.035820,0.074128,0.052457,0.082965,-0.032121,0.042246


In [109]:
market = benchmark

raw_betas = {}

for ticker in peer_group:
    cov = np.cov(return_data[ticker], return_data[market])[0, 1]
    var = np.var(return_data[market])
    raw_betas[ticker] = cov / var

raw_betas = pd.Series(raw_betas, name="raw_betas")
raw_betas


ORCL    1.191630
PLTR    2.113881
PANW    1.224286
CRWD    1.690657
FTNT    1.441189
Name: raw_betas, dtype: float64